# Chapter 3 Computational Lab
## The Concept of Probability

This notebook accompanies Chapter 3 of *Probability Theory with Python and AI*.

Chapter 2 determined which subsets of a sample space are events. We now assign numbers to those events. The purpose of this lab is to make the consequences of the probability axioms computationally visible while keeping a strict distinction among:

- a **mathematical proof**, valid for every probability space;
- an **exact finite verification**, valid for one fully specified finite model;
- an **empirical simulation**, which illustrates behavior under a chosen model but does not prove a theorem.

### Learning goals

By the end of the lab you should be able to:

1. state and check the three Kolmogorov axioms;
2. derive and verify the empty-set, complement, difference and monotonicity rules;
3. distinguish probability zero from impossibility;
4. use the addition rule, Fréchet bounds and finite inclusion--exclusion;
5. apply Boole's inequality without assuming independence;
6. understand continuity from below and above;
7. interpret the first Borel--Cantelli lemma;
8. construct finite and countable probability measures from point masses;
9. distinguish mathematical validity, empirical calibration and model risk;
10. solve the Chevalier de Méré dice problem using a finite equally likely model;
11. audit AI-generated probability arguments.

> **Chapter boundary.** Conditional probability and independence belong to Chapter 4. Repeated-trial examples in this notebook therefore state the complete probability model directly rather than invoking an independence theorem.


## 0. Setup

Exact rational arithmetic is used whenever practical. This lets us verify identities without floating-point roundoff.


In [ ]:
from fractions import Fraction
from itertools import combinations, product
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def power_set(items):
    items = tuple(items)
    return {
        frozenset(choice)
        for r in range(len(items) + 1)
        for choice in combinations(items, r)
    }


def parse_set(text):
    text = text.strip()
    if not text:
        return frozenset()
    return frozenset(int(x.strip()) for x in text.split(",") if x.strip())


def fmt_set(A):
    if not A:
        return r"\varnothing"
    return r"\{" + ",".join(map(str, sorted(A))) + r"\}"


def fmt_fraction(x):
    x = Fraction(x)
    if x.denominator == 1:
        return str(x.numerator)
    return rf"\frac{{{x.numerator}}}{{{x.denominator}}}"


def event_probability(event, mass):
    event = frozenset(event)
    omega = frozenset(mass)
    if not event <= omega:
        raise ValueError("The event contains an outcome outside Ω.")
    return sum((mass[x] for x in event), Fraction(0, 1))


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Exact finite-probability tools are ready."
    "</div>"
))


## 1. From events to probabilities

A measurable space $(\Omega,\mathcal F)$ tells us which statements are included in the mathematical model. A probability measure adds numerical uncertainty:

$$
P:\mathcal F\longrightarrow\mathbb R.
$$

The expression $P(A)$ is meaningful only when $A\in\mathcal F$.

This point is easy to overlook: a probability model does **not** have to assign a probability to every subset of $\Omega$.


In [ ]:
coarse_omega = frozenset({1, 2, 3, 4, 5, 6})
coarse_A = frozenset({1, 2})
coarse_F = {
    frozenset(),
    coarse_A,
    coarse_omega - coarse_A,
    coarse_omega,
}
coarse_P = {
    frozenset(): Fraction(0, 1),
    coarse_A: Fraction(2, 3),
    coarse_omega - coarse_A: Fraction(1, 3),
    coarse_omega: Fraction(1, 1),
}

display(Markdown(
    "Consider a coarse die model that records only whether the result lies "
    "in $A=\\{1,2\\}$ or in $A^c$."
))
display(Math(r"\mathcal F=\{\varnothing,A,A^c,\Omega\}"))
display(Math(r"P(A)=\frac23,\qquad P(A^c)=\frac13"))

singleton = frozenset({1})
display(Markdown(f"Is `{{1}}` an event in this model? **{singleton in coarse_F}**"))
display(Markdown(
    "Hence $P(\\{1\\})$ is not specified by this probability space."
))


## 2. The probability axioms

A probability measure satisfies three requirements.

**Normalization**

$$
P(\Omega)=1.
$$

**Non-negativity**

$$
P(A)\ge0
\qquad
\text{for every }A\in\mathcal F.
$$

**Countable additivity**

For pairwise disjoint events $A_1,A_2,\ldots$,

$$
P\left(\bigcup_{n=1}^{\infty}A_n\right)
=
\sum_{n=1}^{\infty}P(A_n).
$$

Only these three statements are axioms. The familiar computational formulas that follow are consequences.


### A finite three-outcome probability model

Let

$$
\Omega=\{a,b,c\},
$$

and assign

$$
P(\{a\})=0.70,\qquad
P(\{b\})=0.20,\qquad
P(\{c\})=0.10.
$$

For every event $A\subseteq\Omega$, define $P(A)$ as the sum of the masses of the points in $A$.


In [ ]:
three_mass = {
    "a": Fraction(70, 100),
    "b": Fraction(20, 100),
    "c": Fraction(10, 100),
}
three_omega = frozenset(three_mass)
three_events = power_set(three_omega)

assert all(p >= 0 for p in three_mass.values())
assert sum(three_mass.values(), Fraction(0, 1)) == 1

for A in three_events:
    for B in three_events:
        if A.isdisjoint(B):
            assert event_probability(A | B, three_mass) == (
                event_probability(A, three_mass)
                + event_probability(B, three_mass)
            )

display(Math(
    r"P(\{b,c\})="
    + fmt_fraction(event_probability({"b", "c"}, three_mass))
))
display(Markdown(
    "Normalization, non-negativity and disjoint additivity pass exactly "
    "in this finite model."
))


## 3. Elementary consequences of the axioms

The axioms imply

$$
P(\varnothing)=0,
$$

$$
P(A^c)=1-P(A),
$$

and, whenever $A\subseteq B$,

$$
P(B\setminus A)=P(B)-P(A),
$$

$$
P(A)\le P(B).
$$

Consequently,

$$
0\le P(A)\le1.
$$

The logical distinction is important:

> **Axioms:** normalization, non-negativity, countable additivity.  
> **Theorems:** empty-set rule, finite additivity, complement rule, difference rule and monotonicity.


In [ ]:
elem_mass = {
    0: Fraction(65, 100),
    50: Fraction(15, 100),
    100: Fraction(10, 100),
    250: Fraction(7, 100),
    600: Fraction(3, 100),
}
elem_omega = frozenset(elem_mass)
elem_events = power_set(elem_omega)

A_text = widgets.Text(value="0,100,600", description="A")
B_text = widgets.Text(value="0,50,100,250,600", description="B")
elem_output = widgets.Output()


def update_elementary(*_):
    with elem_output:
        clear_output(wait=True)
        try:
            A = parse_set(A_text.value)
            B = parse_set(B_text.value)
        except ValueError:
            display(Markdown("**Use comma-separated integers.**"))
            return

        if A not in elem_events or B not in elem_events:
            display(Markdown(
                "**A and B must be subsets of Ω={0,50,100,250,600}.**"
            ))
            return

        PA = event_probability(A, elem_mass)
        PB = event_probability(B, elem_mass)
        PAc = event_probability(elem_omega - A, elem_mass)

        display(Math(r"P(A)=" + fmt_fraction(PA)))
        display(Math(r"P(A^c)=" + fmt_fraction(PAc) + r"=1-P(A)"))

        if A <= B:
            diff = event_probability(B - A, elem_mass)
            display(Math(
                r"P(B\setminus A)="
                + fmt_fraction(diff)
                + r"=P(B)-P(A)"
            ))
            display(Markdown(f"Monotonicity check: **{PA <= PB}**"))
        else:
            display(Markdown(
                "$A\\nsubseteq B$, so the difference theorem for nested "
                "events is not being invoked."
            ))


for control in (A_text, B_text):
    control.observe(update_elementary, names="value")

display(widgets.VBox([
    widgets.HBox([A_text, B_text]),
    elem_output,
]))
update_elementary()


## 4. Probability zero and probability one

A statement holds **almost surely** when the event on which it is true has probability one.

Probability zero is not the same as impossibility. Likewise, probability one does not force the event to equal the entire sample space.

For example, let

$$
\Omega=\{\omega_0,\omega_1\},
$$

and assign

$$
P(\{\omega_0\})=1,
\qquad
P(\{\omega_1\})=0.
$$

Then $\{\omega_1\}$ is non-empty but has probability zero, while the proper subset $\{\omega_0\}$ has probability one.


In [ ]:
display(Math(r"P(\{\omega_1\})=0,\qquad \{\omega_1\}\ne\varnothing"))
display(Math(r"P(\{\omega_0\})=1,\qquad \{\omega_0\}\ne\Omega"))


## 5. The addition rule

For arbitrary events $A$ and $B$,

$$
P(A\cup B)
=
P(A)+P(B)-P(A\cap B).
$$

The subtraction removes the second copy of the overlap $A\cap B$.


In [ ]:
addition_A = frozenset({0, 100, 600})
addition_B = frozenset({50, 100, 250})

PA = event_probability(addition_A, elem_mass)
PB = event_probability(addition_B, elem_mass)
PI = event_probability(addition_A & addition_B, elem_mass)
PU = event_probability(addition_A | addition_B, elem_mass)

display(Math(r"A=" + fmt_set(addition_A)))
display(Math(r"B=" + fmt_set(addition_B)))
display(Math(r"A\cap B=" + fmt_set(addition_A & addition_B)))
display(Math(
    r"P(A\cup B)="
    + fmt_fraction(PA)
    + "+"
    + fmt_fraction(PB)
    + "-"
    + fmt_fraction(PI)
    + "="
    + fmt_fraction(PU)
))


### The three disjoint pieces of the union

The union can be written as the disjoint union of

$$
A\setminus B,\qquad
A\cap B,\qquad
B\setminus A.
$$

The next plot shows how much probability mass each piece carries in the finite model above.


In [ ]:
left = event_probability(addition_A - addition_B, elem_mass)
middle = event_probability(addition_A & addition_B, elem_mass)
right = event_probability(addition_B - addition_A, elem_mass)

values = [float(left), float(middle), float(right)]
labels = [r"$A\setminus B$", r"$A\cap B$", r"$B\setminus A$"]

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(labels, values)
ax.set_ylabel("probability mass")
ax.set_title("Three disjoint pieces of the union")
for i, value in enumerate(values):
    ax.text(i, value + 0.01, f"{value:.2f}", ha="center")
ax.set_ylim(0, max(values) + 0.12)
plt.show()

display(Math(
    r"P(A\cup B)"
    r"=P(A\setminus B)+P(A\cap B)+P(B\setminus A)"
))


## 6. Fréchet bounds

When only the marginal probabilities $P(A)=a$ and $P(B)=b$ are known,

$$
\max\{0,a+b-1\}
\le
P(A\cap B)
\le
\min\{a,b\}.
$$

These bounds are sharp.

For any feasible value $x=P(A\cap B)$, the four-point model

$$
p_{11}=x,\qquad
p_{10}=a-x,\qquad
p_{01}=b-x,\qquad
p_{00}=1-a-b+x
$$

has the desired marginals and intersection.


In [ ]:
frechet_a = widgets.FloatSlider(
    value=0.70, min=0.0, max=1.0, step=0.01, description="P(A)"
)
frechet_b = widgets.FloatSlider(
    value=0.60, min=0.0, max=1.0, step=0.01, description="P(B)"
)
frechet_x = widgets.FloatSlider(
    value=0.30, min=0.0, max=1.0, step=0.01, description="P(A∩B)"
)
frechet_output = widgets.Output()


def update_frechet(*_):
    with frechet_output:
        clear_output(wait=True)
        a = frechet_a.value
        b = frechet_b.value
        x = frechet_x.value

        lower = max(0.0, a + b - 1.0)
        upper = min(a, b)
        feasible = lower - 1e-12 <= x <= upper + 1e-12

        display(Math(rf"{lower:.2f}\le P(A\cap B)\le {upper:.2f}"))
        display(Markdown(f"Chosen intersection value feasible? **{feasible}**"))

        masses = [x, a - x, b - x, 1 - a - b + x]
        labels = ["11", "10", "01", "00"]

        if feasible:
            display(Markdown(
                "**Four-point masses:** "
                + ", ".join(
                    f"$p_{{{lab}}}={mass:.2f}$"
                    for lab, mass in zip(labels, masses)
                )
            ))

            fig, ax = plt.subplots(figsize=(7, 3))
            ax.bar(labels, masses)
            ax.set_ylabel("probability mass")
            ax.set_title("Sharpness construction")
            ax.set_ylim(0, max(masses + [0.1]) + 0.1)
            plt.show()
        else:
            display(Markdown(
                "The selected intersection is impossible: at least one "
                "of the four masses becomes negative."
            ))


for control in (frechet_a, frechet_b, frechet_x):
    control.observe(update_frechet, names="value")

display(widgets.VBox([
    widgets.HBox([frechet_a, frechet_b]),
    frechet_x,
    frechet_output,
]))
update_frechet()


## 7. Finite inclusion--exclusion

For three events,

$$
P(A\cup B\cup C)
=
P(A)+P(B)+P(C)
-
P(A\cap B)-P(A\cap C)-P(B\cap C)
+
P(A\cap B\cap C).
$$

Choose uniformly from $\Omega=\{1,\ldots,12\}$ and define:

- $A$: the integer is divisible by $2$;
- $B$: the integer is divisible by $3$;
- $C$: the integer is divisible by $4$.


In [ ]:
ie_omega = frozenset(range(1, 13))
A = frozenset(x for x in ie_omega if x % 2 == 0)
B = frozenset(x for x in ie_omega if x % 3 == 0)
C = frozenset(x for x in ie_omega if x % 4 == 0)


def uniform_prob(event, omega):
    return Fraction(len(event), len(omega))


inclusion_exclusion_value = (
    uniform_prob(A, ie_omega)
    + uniform_prob(B, ie_omega)
    + uniform_prob(C, ie_omega)
    - uniform_prob(A & B, ie_omega)
    - uniform_prob(A & C, ie_omega)
    - uniform_prob(B & C, ie_omega)
    + uniform_prob(A & B & C, ie_omega)
)

display(Math(r"|A|=6,\qquad |B|=4,\qquad |C|=3"))
display(Math(
    r"|A\cap B|=2,\qquad |A\cap C|=3,\qquad |B\cap C|=1"
))
display(Math(r"|A\cap B\cap C|=1"))
display(Math(
    r"P(A\cup B\cup C)=" + fmt_fraction(inclusion_exclusion_value)
))
display(Markdown(
    "Direct enumeration agrees: "
    f"**{uniform_prob(A | B | C, ie_omega) == inclusion_exclusion_value}**"
))


## 8. Boole's inequality: the union bound

For any sequence of events,

$$
P\left(\bigcup_{n=1}^{\infty}A_n\right)
\le
\sum_{n=1}^{\infty}P(A_n).
$$

No independence assumption is required.

If $40$ events each have probability at most $0.002$, then

$$
P\left(\bigcup_{i=1}^{40}A_i\right)\le0.08.
$$

The bound can be strict because overlaps reduce the probability of the union.


In [ ]:
rare_m = widgets.IntSlider(
    value=40, min=1, max=200, description="events m"
)
rare_p = widgets.FloatSlider(
    value=0.002, min=0.0, max=0.05, step=0.001, description="max P(Aᵢ)"
)
rare_output = widgets.Output()


def update_union_bound(*_):
    with rare_output:
        clear_output(wait=True)
        m = rare_m.value
        p = rare_p.value
        raw = m * p
        useful = min(1.0, raw)

        display(Math(
            rf"P\left(\bigcup_{{i=1}}^{{{m}}}A_i\right)"
            rf"\le\sum_{{i=1}}^{{{m}}}P(A_i)"
            rf"\le {m}({p:.3f})={raw:.3f}"
        ))
        display(Markdown(
            f"Usable probability upper bound: **{useful:.3f}**."
        ))


for control in (rare_m, rare_p):
    control.observe(update_union_bound, names="value")

display(widgets.VBox([
    widgets.HBox([rare_m, rare_p]),
    rare_output,
]))
update_union_bound()


### A lower bound for simultaneous success

Applying Boole's inequality to the complements gives

$$
P\left(\bigcap_{i=1}^{m}A_i\right)
\ge
1-\sum_{i=1}^{m}P(A_i^c).
$$

Thus if five requirements each fail with probability at most $0.01$,

$$
P\left(\bigcap_{i=1}^{5}A_i\right)\ge0.95.
$$


In [ ]:
sim_m = widgets.IntSlider(
    value=5, min=1, max=50, description="requirements"
)
sim_fail = widgets.FloatSlider(
    value=0.01, min=0.0, max=0.20, step=0.005, description="max failure"
)
sim_output = widgets.Output()


def update_simultaneous(*_):
    with sim_output:
        clear_output(wait=True)
        raw = 1 - sim_m.value * sim_fail.value
        useful = max(0.0, raw)

        display(Math(
            rf"P\left(\bigcap_{{i=1}}^{{{sim_m.value}}}A_i\right)"
            rf"\ge1-{sim_m.value}({sim_fail.value:.3f})"
            rf"={raw:.3f}"
        ))
        display(Markdown(
            f"Usable lower bound after truncating at $0$: **{useful:.3f}**."
        ))


for control in (sim_m, sim_fail):
    control.observe(update_simultaneous, names="value")

display(widgets.VBox([
    widgets.HBox([sim_m, sim_fail]),
    sim_output,
]))
update_simultaneous()


## 9. Continuity of probability

If

$$
A_n\uparrow A,
$$

then

$$
P(A_n)\longrightarrow P(A).
$$

If

$$
A_n\downarrow A,
$$

then

$$
P(A_n)\longrightarrow P(A).
$$

This result lets us approximate complicated limiting events by monotone sequences of simpler events.


### A countable model

Take

$$
\Omega=\{1,2,3,\ldots\},
\qquad
P(\{n\})=2^{-n}.
$$

Define

$$
A_n=\{1,\ldots,n\},
$$

and

$$
C_n=\{n,n+1,\ldots\}.
$$

Then $A_n\uparrow\Omega$ and $C_n\downarrow\varnothing$.


In [ ]:
continuity_N = widgets.IntSlider(
    value=12, min=2, max=30, description="max n"
)
continuity_output = widgets.Output()


def update_continuity(*_):
    with continuity_output:
        clear_output(wait=True)
        N = continuity_N.value
        ns = np.arange(1, N + 1)

        increasing = np.array(
            [1 - 2 ** (-int(n)) for n in ns],
            dtype=float,
        )
        decreasing = np.array(
            [2 ** (-(int(n) - 1)) for n in ns],
            dtype=float,
        )

        display(Math(
            rf"P(A_{{{N}}})=1-2^{{-{N}}}\approx {increasing[-1]:.8f}"
        ))
        display(Math(
            rf"P(C_{{{N}}})=2^{{-({N}-1)}}\approx {decreasing[-1]:.8f}"
        ))

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.plot(ns, increasing, marker="o", label=r"$P(A_n)$")
        ax.plot(ns, decreasing, marker="o", label=r"$P(C_n)$")
        ax.axhline(1, linestyle="--")
        ax.axhline(0, linestyle="--")
        ax.set_xlabel("n")
        ax.set_ylabel("probability")
        ax.set_ylim(-0.03, 1.03)
        ax.legend()
        ax.set_title("Continuity from below and above")
        plt.show()


continuity_N.observe(update_continuity, names="value")
display(widgets.VBox([continuity_N, continuity_output]))
update_continuity()


### Vanishing tails

If $B_1,B_2,\ldots$ are pairwise disjoint, then

$$
P\left(\bigcup_{k=n}^{\infty}B_k\right)\longrightarrow0.
$$

For the geometric point-mass model $P(\{k\})=2^{-k}$, the tail probability is exactly $2^{-(n-1)}$.


In [ ]:
tail_n = widgets.IntSlider(
    value=8, min=1, max=25, description="n"
)
tail_output = widgets.Output()


def update_tail(*_):
    with tail_output:
        clear_output(wait=True)
        n = tail_n.value
        tail = 2 ** (-(n - 1))

        display(Math(
            rf"P(R_{{{n}}})=2^{{-({n}-1)}}={tail:.8f}"
        ))
        display(Math(
            rf"R_{{{n}}}=\bigcup_{{k={n}}}^{{\infty}}B_k"
        ))


tail_n.observe(update_tail, names="value")
display(widgets.VBox([tail_n, tail_output]))
update_tail()


## 10. First Borel--Cantelli lemma

If

$$
\sum_{n=1}^{\infty}P(A_n)<\infty,
$$

then

$$
P\left(\limsup_{n\to\infty}A_n\right)=0.
$$

Equivalently, with probability one only finitely many of the events $A_n$ occur.

The proof uses the tail unions

$$
B_n=\bigcup_{k=n}^{\infty}A_k,
$$

together with continuity from above and Boole's inequality:

$$
P(B_n)
\le
\sum_{k=n}^{\infty}P(A_k).
$$

No independence assumption is used.


In [ ]:
bc_base = widgets.FloatSlider(
    value=3.0, min=1.1, max=6.0, step=0.1, description="base b"
)
bc_N = widgets.IntSlider(
    value=12, min=2, max=30, description="max n"
)
bc_output = widgets.Output()


def update_bc(*_):
    with bc_output:
        clear_output(wait=True)
        b = bc_base.value
        N = bc_N.value
        ns = np.arange(1, N + 1)

        tail_bounds = np.array([
            (b ** (-(int(n) - 1))) / (b - 1)
            for n in ns
        ])

        display(Math(
            rf"\sum_{{n=1}}^\infty {b:.1f}^{{-n}}"
            rf"=\frac{{1}}{{{b:.1f}-1}}"
            rf"\approx {1/(b-1):.6f}"
        ))

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.plot(ns, tail_bounds, marker="o")
        ax.set_xlabel("tail starting index n")
        ax.set_ylabel("tail-sum upper bound")
        ax.set_title(
            r"Upper bound for $P(\bigcup_{k=n}^{\infty}A_k)$"
        )
        plt.show()

        display(Markdown(
            "The tail of a convergent non-negative series tends to zero. "
            "That is the numerical mechanism behind the proof."
        ))


for control in (bc_base, bc_N):
    control.observe(update_bc, names="value")

display(widgets.VBox([
    widgets.HBox([bc_base, bc_N]),
    bc_output,
]))
update_bc()


## 11. Finite and countable probability models

On a finite or countably infinite sample space with $\mathcal F=\mathcal P(\Omega)$, a probability measure is completely determined by point masses.

If

$$
p_i\ge0,
\qquad
\sum_i p_i=1,
$$

then

$$
P(A)
=
\sum_{\{i:\omega_i\in A\}}p_i.
$$

Conversely, every probability measure on the full power set has this form.


### Normalizing proportional masses

Suppose the masses of the values $0,1,2,3$ are proportional to

$$
1,2,3,4.
$$

Normalization determines the constant of proportionality.


In [ ]:
weights_text = widgets.Text(
    value="1,2,3,4",
    description="weights",
)
norm_output = widgets.Output()


def update_normalization(*_):
    with norm_output:
        clear_output(wait=True)

        try:
            weights = [
                Fraction(x.strip())
                for x in weights_text.value.split(",")
                if x.strip()
            ]
        except Exception:
            display(Markdown(
                "**Enter comma-separated non-negative numbers.**"
            ))
            return

        if not weights or any(w < 0 for w in weights):
            display(Markdown(
                "**Weights must be non-negative and the list non-empty.**"
            ))
            return

        total = sum(weights, Fraction(0, 1))
        if total == 0:
            display(Markdown("**At least one weight must be positive.**"))
            return

        probabilities = [w / total for w in weights]

        display(Math(r"\sum_i w_i=" + fmt_fraction(total)))
        for i, p in enumerate(probabilities):
            display(Math(rf"p_{i}=" + fmt_fraction(p)))
        display(Math(r"\sum_i p_i=1"))


weights_text.observe(update_normalization, names="value")
display(widgets.VBox([weights_text, norm_output]))
update_normalization()


### A geometric count model

For $0<q<1$, define on $\Omega=\{0,1,2,\ldots\}$

$$
P(\{n\})=(1-q)q^n.
$$

Since

$$
\sum_{n=0}^{\infty}(1-q)q^n=1,
$$

these point masses define a probability measure.

For $r\ge0$,

$$
P(\{r,r+1,\ldots\})=q^r.
$$


In [ ]:
geom_q = widgets.FloatSlider(
    value=0.60, min=0.05, max=0.95, step=0.05, description="q"
)
geom_r = widgets.IntSlider(
    value=4, min=0, max=20, description="r"
)
geom_output = widgets.Output()


def update_geom(*_):
    with geom_output:
        clear_output(wait=True)
        q = geom_q.value
        r = geom_r.value

        display(Math(
            rf"P(N\ge {r})={q:.2f}^{{{r}}}={q**r:.6f}"
        ))

        ns = np.arange(0, min(20, r + 12))
        pmf = (1 - q) * q ** ns

        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.bar(ns, pmf)
        ax.set_xlabel("n")
        ax.set_ylabel(r"$P(\{n\})$")
        ax.set_title("Geometric point masses")
        plt.show()


for control in (geom_q, geom_r):
    control.observe(update_geom, names="value")

display(widgets.VBox([
    widgets.HBox([geom_q, geom_r]),
    geom_output,
]))
update_geom()


## 12. The equally likely finite model

If $\Omega$ is finite, $\mathcal F=\mathcal P(\Omega)$, and every singleton has the same probability, then

$$
P(A)=\frac{|A|}{|\Omega|}.
$$

This formula is **not** a separate axiom. It follows from the probability axioms plus the modelling assumption that all elementary outcomes are equally likely.


In [ ]:
die_pairs = list(product(range(1, 7), repeat=2))
sum_at_least_10 = [pair for pair in die_pairs if sum(pair) >= 10]

display(Markdown(f"Total ordered pairs: **{len(die_pairs)}**"))
display(Markdown(
    f"Favourable ordered pairs: **{sum_at_least_10}**"
))
display(Math(r"P(\text{sum}\ge10)=\frac{6}{36}=\frac16"))


## 13. Loaded die, calibration and model risk

A physical die need not be fair. A one-throw model may use

$$
p_i\ge0,
\qquad
\sum_{i=1}^{6}p_i=1.
$$

Observed relative frequencies can help assess whether a proposed probability vector is empirically plausible. But the **probability axioms alone do not imply convergence of empirical frequencies**.

This distinction separates three questions:

- Is the probability vector mathematically valid?
- Is it well calibrated to observed data?
- Is a fixed probability vector scientifically appropriate if the mechanism changes over time?


In [ ]:
loaded_probs = np.array([0.30, 0.12, 0.14, 0.16, 0.15, 0.13])
assert abs(loaded_probs.sum() - 1.0) < 1e-12

loaded_N = widgets.IntSlider(
    value=500, min=20, max=5000, step=20, description="throws"
)
loaded_seed = widgets.IntSlider(
    value=7, min=0, max=100, description="seed"
)
loaded_output = widgets.Output()


def update_loaded(*_):
    with loaded_output:
        clear_output(wait=True)

        rng = np.random.default_rng(loaded_seed.value)
        draws = rng.choice(
            np.arange(1, 7),
            size=loaded_N.value,
            p=loaded_probs,
        )
        frequencies = np.array([
            (draws == i).mean()
            for i in range(1, 7)
        ])

        x = np.arange(1, 7)
        width = 0.36

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.bar(
            x - width / 2,
            loaded_probs,
            width=width,
            label="model probabilities",
        )
        ax.bar(
            x + width / 2,
            frequencies,
            width=width,
            label="empirical frequencies",
        )
        ax.set_xticks(x)
        ax.set_xlabel("face")
        ax.set_ylabel("probability / frequency")
        ax.set_title("One simulated sample from a loaded-die model")
        ax.legend()
        plt.show()

        display(Markdown(
            "This is a simulation under the selected model. It does not "
            "prove a law of large numbers, and it does not establish that "
            "a physical die follows these probabilities."
        ))


for control in (loaded_N, loaded_seed):
    control.observe(update_loaded, names="value")

display(widgets.VBox([
    widgets.HBox([loaded_N, loaded_seed]),
    loaded_output,
]))
update_loaded()


### Structural change

Suppose the die mechanism changes halfway through the observation period.

Pooling the data can produce one apparently reasonable empirical probability vector, even though no single vector describes both regimes. This is a simple illustration of **model risk**.


In [ ]:
regime_p1 = np.array([0.42, 0.12, 0.12, 0.12, 0.11, 0.11])
regime_p2 = np.array([0.12, 0.12, 0.12, 0.12, 0.20, 0.32])

change_N = widgets.IntSlider(
    value=1000, min=100, max=5000, step=100, description="total throws"
)
change_output = widgets.Output()


def update_change(*_):
    with change_output:
        clear_output(wait=True)

        N = change_N.value
        n1 = N // 2
        n2 = N - n1

        rng = np.random.default_rng(2026)
        d1 = rng.choice(np.arange(1, 7), size=n1, p=regime_p1)
        d2 = rng.choice(np.arange(1, 7), size=n2, p=regime_p2)
        pooled_draws = np.concatenate([d1, d2])

        f1 = np.array([(d1 == i).mean() for i in range(1, 7)])
        f2 = np.array([(d2 == i).mean() for i in range(1, 7)])
        pooled = np.array([
            (pooled_draws == i).mean()
            for i in range(1, 7)
        ])

        x = np.arange(1, 7)
        width = 0.25

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.bar(x - width, f1, width=width, label="first half")
        ax.bar(x, pooled, width=width, label="pooled")
        ax.bar(x + width, f2, width=width, label="second half")
        ax.set_xticks(x)
        ax.set_xlabel("face")
        ax.set_ylabel("empirical frequency")
        ax.set_title("Pooling can hide a changing mechanism")
        ax.legend()
        plt.show()


change_N.observe(update_change, names="value")
display(widgets.VBox([change_N, change_output]))
update_change()


## 14. Historical problem: Chevalier de Méré's dice

Compare two even-money bets:

1. obtain at least one six in four throws of one fair die;
2. obtain at least one double six in twenty-four throws of two fair dice.

For each game, the probability model states that every complete ordered sequence of outcomes is equally likely.

For the first bet,

$$
P(A)
=
1-\left(\frac56\right)^4
=
\frac{671}{1296}
\approx0.517747.
$$

For the second bet,

$$
P(B)
=
1-\left(\frac{35}{36}\right)^{24}
\approx0.491404.
$$

Thus the first bet is favorable and the second is unfavorable.

No theorem about independent events is required: the complete-sequence probability model is specified directly.


In [ ]:
de_mere_one = 1 - Fraction(5, 6) ** 4
de_mere_two = 1 - Fraction(35, 36) ** 24

display(Math(
    r"P(A)=1-\left(\frac56\right)^4="
    + fmt_fraction(de_mere_one)
    + rf"\approx {float(de_mere_one):.6f}"
))
display(Math(
    r"P(B)=1-\left(\frac{35}{36}\right)^{24}"
    + rf"\approx {float(de_mere_two):.6f}"
))
display(Markdown(
    f"First bet above $1/2$: **{de_mere_one > Fraction(1, 2)}**"
))
display(Markdown(
    f"Second bet below $1/2$: **{de_mere_two < Fraction(1, 2)}**"
))


### The incorrect proportionality argument

It is tempting to write

$$
4\left(\frac16\right)
=
24\left(\frac1{36}\right)
=
\frac23.
$$

But this counts opportunities for success, not the probability of **at least one** success. A complete sequence containing several successes is counted repeatedly.

The sum of individual probabilities is generally only an upper bound:

$$
P\left(\bigcup_i A_i\right)
\le
\sum_iP(A_i).
$$


In [ ]:
display(Math(
    r"4\cdot\frac16=24\cdot\frac1{36}=\frac23"
))
display(Math(
    r"P(A)=" + fmt_fraction(de_mere_one) + r"\ne\frac23"
))
display(Math(
    r"P(B)\approx"
    + f"{float(de_mere_two):.6f}"
    + r"\ne\frac23"
))


## 15. Only countably many points can have positive mass

Suppose every singleton is measurable. Define

$$
D=\{\omega:P(\{\omega\})>0\}.
$$

For $n\ge1$, let

$$
D_n
=
\left\{
\omega:P(\{\omega\})\ge\frac1n
\right\}.
$$

Each $D_n$ contains at most $n$ points; otherwise the probability of the union of $n+1$ such singletons would exceed one.

Since every positive number is at least $1/n$ for some $n$,

$$
D=\bigcup_{n=1}^{\infty}D_n.
$$

Therefore $D$ is at most countable.


### A null point beside countably many positive-mass points

Let

$$
\Omega=\{1,2,3,\ldots\}\cup\{\ast\},
$$

and define

$$
P(\{n\})=2^{-n},
\qquad
P(\{\ast\})=0.
$$

The positive masses sum to one, while $\{\ast\}$ is a non-empty event of probability zero.


In [ ]:
partial_N = widgets.IntSlider(
    value=10, min=1, max=30, description="N"
)
partial_output = widgets.Output()


def update_partial_mass(*_):
    with partial_output:
        clear_output(wait=True)

        N = partial_N.value
        partial = sum(
            Fraction(1, 2 ** n)
            for n in range(1, N + 1)
        )
        tail = Fraction(1, 2 ** N)

        display(Math(
            rf"\sum_{{n=1}}^{{{N}}}2^{{-n}}="
            + fmt_fraction(partial)
        ))
        display(Math(
            rf"1-\sum_{{n=1}}^{{{N}}}2^{{-n}}="
            + fmt_fraction(tail)
        ))
        display(Math(r"P(\{\ast\})=0"))


partial_N.observe(update_partial_mass, names="value")
display(widgets.VBox([partial_N, partial_output]))
update_partial_mass()


## 16. Capped positive-part transformations

For

$$
C=\min\{(X-d)^+,u\},
\qquad
x^+=\max\{x,0\},
$$

the event structure is

$$
\{C=0\}=\{X\le d\},
$$

$$
\{0<C<u\}=\{d<X<d+u\},
$$

and

$$
\{C=u\}=\{X\ge d+u\}.
$$

The next widget applies this transformation to a finite model.


In [ ]:
cap_mass = {
    0: Fraction(50, 100),
    100: Fraction(25, 100),
    300: Fraction(15, 100),
    800: Fraction(10, 100),
}

cap_d = widgets.IntSlider(
    value=200, min=0, max=600, step=50, description="d"
)
cap_u = widgets.IntSlider(
    value=400, min=50, max=600, step=50, description="u"
)
cap_output = widgets.Output()


def update_cap(*_):
    with cap_output:
        clear_output(wait=True)

        d = cap_d.value
        u = cap_u.value

        transformed = {
            x: min(max(x - d, 0), u)
            for x in cap_mass
        }

        output_mass = {}
        for x, c_value in transformed.items():
            output_mass[c_value] = (
                output_mass.get(c_value, Fraction(0, 1))
                + cap_mass[x]
            )

        display(Markdown("**Transformation table:**"))
        for x in sorted(transformed):
            display(Markdown(
                f"- $X={x}$ gives $C={transformed[x]}$"
            ))

        display(Markdown("**Resulting probability masses:**"))
        for c_value in sorted(output_mass):
            display(Math(
                rf"P(C={c_value})="
                + fmt_fraction(output_mass[c_value])
            ))


for control in (cap_d, cap_u):
    control.observe(update_cap, names="value")

display(widgets.VBox([
    widgets.HBox([cap_d, cap_u]),
    cap_output,
]))
update_cap()


## 17. Exact finite computational verification

On a finite space Python can enumerate every event and check the arithmetic consequences of a proposed point-mass model.

This is an exact verification **inside that finite model**. It is not a substitute for a general mathematical proof.


In [ ]:
verify_omega = frozenset({0, 50, 100, 250, 600})
verify_events = power_set(verify_omega)
verify_mass = {
    0: Fraction(65, 100),
    50: Fraction(15, 100),
    100: Fraction(10, 100),
    250: Fraction(7, 100),
    600: Fraction(3, 100),
}


def verify_prob(event):
    event = frozenset(event)
    if event not in verify_events:
        raise ValueError("Argument must belong to P(Ω).")
    return event_probability(event, verify_mass)


assert len(verify_events) == 2 ** len(verify_omega)
assert all(p >= 0 for p in verify_mass.values())
assert sum(verify_mass.values(), Fraction(0, 1)) == 1
assert verify_prob(verify_omega) == 1
assert verify_prob(frozenset()) == 0

for A in verify_events:
    assert 0 <= verify_prob(A) <= 1
    assert verify_prob(verify_omega - A) == 1 - verify_prob(A)

    for B in verify_events:
        if A <= B:
            assert verify_prob(A) <= verify_prob(B)
            assert verify_prob(B - A) == verify_prob(B) - verify_prob(A)

        if A.isdisjoint(B):
            assert verify_prob(A | B) == verify_prob(A) + verify_prob(B)

        assert verify_prob(A | B) == (
            verify_prob(A)
            + verify_prob(B)
            - verify_prob(A & B)
        )

show_result(
    "Exact exhaustive verification passed",
    r"P(\Omega)=1",
    r"P(A^c)=1-P(A)",
    r"A\subseteq B\Longrightarrow P(A)\le P(B)",
    r"P(A\cup B)=P(A)+P(B)-P(A\cap B)",
    note=(
        f"All {len(verify_events)} events and "
        f"{len(verify_events) ** 2} ordered event pairs were checked."
    ),
)


## 18. Guided exercise generator

Solve each problem before pressing **Hint** or **Reveal**.


In [ ]:
rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Complement", "complement"),
        ("Addition rule", "addition"),
        ("Fréchet bounds", "frechet"),
        ("Union bound", "boole"),
        ("Point-mass normalization", "mass"),
        ("Continuity", "continuity"),
        ("Borel–Cantelli", "bc"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
answer_box = widgets.Text(description="Answer")
check_button = widgets.Button(description="Check")
exercise_prompt = widgets.Output()
exercise_feedback = widgets.Output()
exercise_state = {}


def make_exercise(_=None):
    kind = exercise_kind.value
    if kind == "random":
        kind = rng.choice([
            "complement",
            "addition",
            "frechet",
            "boole",
            "mass",
            "continuity",
            "bc",
        ])

    if kind == "complement":
        pa = rng.choice([0.20, 0.30, 0.40, 0.65, 0.75])
        answer = 1 - pa
        prompt = f"If P(A)={pa:.2f}, find P(Aᶜ)."
        hint = "Use the complement rule."
        solution = rf"P(A^c)=1-P(A)={answer:.2f}."

    elif kind == "addition":
        answer = 0.80
        prompt = (
            "If P(A)=0.55, P(B)=0.40 and P(A∩B)=0.15, "
            "find P(A∪B)."
        )
        hint = "Subtract the overlap once."
        solution = r"P(A\cup B)=0.55+0.40-0.15=0.80."

    elif kind == "frechet":
        answer = "0.3,0.6"
        prompt = (
            "If P(A)=0.7 and P(B)=0.6, enter the Fréchet interval "
            "for P(A∩B) as lower,upper."
        )
        hint = "Lower=max{0,a+b−1}; upper=min{a,b}."
        solution = r"0.3\le P(A\cap B)\le0.6."

    elif kind == "boole":
        m = rng.choice([20, 25, 40, 50])
        p = rng.choice([0.001, 0.002, 0.005, 0.01])
        answer = min(1.0, m * p)
        prompt = (
            f"{m} events each have probability at most {p:.3f}. "
            "Give the union-bound upper bound."
        )
        hint = "Add the individual upper bounds."
        solution = rf"P(\cup_iA_i)\le {m}({p:.3f})={m*p:.3f}."

    elif kind == "mass":
        answer = 0.70
        prompt = (
            "Masses on 0,1,2,3 are proportional to 1,2,3,4. "
            "Find P(X≥2)."
        )
        hint = "Normalize by 1+2+3+4=10."
        solution = r"P(X\ge2)=\frac3{10}+\frac4{10}=0.70."

    elif kind == "continuity":
        answer = 0.0
        prompt = "If A_n↓∅, what is lim P(A_n)?"
        hint = "Use continuity from above."
        solution = (
            r"A_n\downarrow\varnothing"
            r"\Longrightarrow P(A_n)\to P(\varnothing)=0."
        )

    else:
        answer = 0.0
        prompt = (
            "If ∑P(A_n)<∞, what is P(limsup A_n) according to "
            "the first Borel–Cantelli lemma?"
        )
        hint = "Only finitely many A_n occur almost surely."
        solution = r"P(\limsup_nA_n)=0."

    exercise_state.clear()
    exercise_state.update(
        answer=str(answer).lower(),
        hint=hint,
        solution=solution,
    )
    answer_box.value = ""

    with exercise_prompt:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))

    with exercise_feedback:
        clear_output(wait=True)


def show_hint(_):
    with exercise_feedback:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + exercise_state["hint"]))


def reveal_solution(_):
    with exercise_feedback:
        clear_output(wait=True)
        display(Math(exercise_state["solution"]))


def check_answer(_):
    with exercise_feedback:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ", "")
        target = exercise_state["answer"].replace(" ", "")

        try:
            if "," not in target and abs(float(guess) - float(target)) < 1e-9:
                display(Markdown("**Correct.**"))
                return
        except Exception:
            pass

        if guess == target:
            display(Markdown("**Correct.**"))
        else:
            display(Markdown(
                "**Not yet.** Identify the exact theorem before doing "
                "the arithmetic."
            ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal_solution)
check_button.on_click(check_answer)

display(widgets.VBox([
    widgets.HBox([exercise_kind, new_button]),
    exercise_prompt,
    widgets.HBox([answer_box, check_button]),
    widgets.HBox([hint_button, reveal_button]),
    exercise_feedback,
]))

make_exercise()


## 19. AI Audit: probability axioms and consequences

An AI system can perform correct arithmetic while silently using the wrong mathematical model. Check every answer against the following protocol.

1. **Domain:** Is every set whose probability is used actually in $\mathcal F$?
2. **Normalization:** Does the proposed model satisfy $P(\Omega)=1$?
3. **Non-negativity:** Are all assigned probabilities non-negative?
4. **Additivity:** Is simple addition being applied only to disjoint events?
5. **Overlap:** If events overlap, has the intersection been subtracted correctly?
6. **Infinite spaces:** Has finite additivity been mistaken for countable additivity?
7. **Null events:** Is the AI falsely concluding that $P(A)=0$ implies $A=\varnothing$?
8. **Union bounds:** Is $\sum_iP(A_i)$ being treated as an equality without justification?
9. **Continuity:** Are the events genuinely increasing or decreasing?
10. **Borel--Cantelli:** Is independence being incorrectly added as a hypothesis of the first lemma?
11. **Uniform models:** If $P(A)=|A|/|\Omega|$ is used, were equally likely elementary outcomes specified?
12. **Data versus model:** Is empirical agreement being confused with mathematical validity?
13. **Chapter boundary:** Is conditional probability or independence being invoked before Chapter 4 when a direct finite-model argument is available?

### Claims to audit

- “Finite additivity is enough to define probability on an infinite sample space.”
- “For arbitrary events $A$ and $B$, $P(A\cup B)=P(A)+P(B)$.”
- “If $P(A)=0$, then $A$ is impossible.”

Each statement is false without further qualification.


### Suggested AI audit prompts

- “Derive $P(A^c)=1-P(A)$ from the Kolmogorov axioms.”
- “Construct a four-point model that attains both Fréchet endpoints for given marginals.”
- “Explain why Boole's inequality does not require independence.”
- “Prove the first Borel--Cantelli lemma from tail unions and continuity from above.”
- “Solve the Chevalier de Méré problem using only a fully specified finite equally likely model.”
- “Give a non-empty probability-zero event.”
- “Separate mathematical validity of a loaded-die model from empirical adequacy and model risk.”


## 20. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. Which is an axiom of probability?",
        [
            "Choose...",
            "P(∅)=0",
            "P(Aᶜ)=1−P(A)",
            "countable additivity",
            "monotonicity",
        ],
        "countable additivity",
        r"\text{Countable additivity is an axiom; the other listed rules are derived.}",
    ),
    (
        "2. If A⊆B, which must hold?",
        [
            "Choose...",
            "P(A)≥P(B)",
            "P(A)≤P(B)",
            "P(A)=P(B)",
            "A=B",
        ],
        "P(A)≤P(B)",
        r"A\subseteq B\Longrightarrow P(A)\le P(B).",
    ),
    (
        "3. For arbitrary A and B:",
        [
            "Choose...",
            "P(A∪B)=P(A)+P(B)",
            "P(A∪B)=P(A)+P(B)−P(A∩B)",
            "P(A∪B)=P(A∩B)",
        ],
        "P(A∪B)=P(A)+P(B)−P(A∩B)",
        r"P(A\cup B)=P(A)+P(B)-P(A\cap B).",
    ),
    (
        "4. Boole's inequality requires independence:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{The union bound is valid under arbitrary dependence.}",
    ),
    (
        "5. If A_n↓∅, then:",
        [
            "Choose...",
            "P(A_n)→1",
            "P(A_n)→0",
            "P(A_n) is constant",
            "no conclusion",
        ],
        "P(A_n)→0",
        r"A_n\downarrow\varnothing\Longrightarrow P(A_n)\to0.",
    ),
    (
        "6. If ∑P(A_n)<∞, the first Borel–Cantelli lemma says:",
        [
            "Choose...",
            "all A_n occur",
            "infinitely many occur a.s.",
            "only finitely many occur a.s.",
            "the A_n are independent",
        ],
        "only finitely many occur a.s.",
        r"P(\limsup_nA_n)=0.",
    ),
    (
        "7. P(A)=0 implies A=∅:",
        ["Choose...", "always true", "false in general"],
        "false in general",
        r"\text{A non-empty null event can occur.}",
    ),
    (
        "8. P(A)=|A|/|Ω| requires:",
        [
            "Choose...",
            "only finiteness",
            "equally likely elementary outcomes",
            "independence",
            "continuity",
        ],
        "equally likely elementary outcomes",
        r"\text{Uniform singleton masses are an additional modelling assumption.}",
    ),
    (
        "9. In de Méré's first bet the success probability is:",
        ["Choose...", "<1/2", "=1/2", ">1/2"],
        ">1/2",
        r"1-(5/6)^4\approx0.517747.",
    ),
    (
        "10. A mathematically valid probability vector is automatically a good physical model:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{Mathematical validity and empirical adequacy are distinct.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="340px"),
    )
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:660px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_, _, correct, _) in zip(
                quiz_widgets,
                quiz_data,
            )
        )
        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))

        for i, (
            widget,
            (_, _, correct, explanation),
        ) in enumerate(zip(quiz_widgets, quiz_data), 1):
            mark = "✓" if widget.value == correct else "✗"
            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))
            display(Math(explanation))


grade_button.on_click(grade_quiz)
display(widgets.VBox(
    quiz_rows + [grade_button, quiz_output]
))


## 21. Automatic mathematical verification

The final code cell checks representative identities from the chapter with exact arithmetic.


In [ ]:
omega = frozenset({0, 50, 100, 250, 600})
events = power_set(omega)
mass = {
    0: Fraction(65, 100),
    50: Fraction(15, 100),
    100: Fraction(10, 100),
    250: Fraction(7, 100),
    600: Fraction(3, 100),
}


def P(A):
    return event_probability(frozenset(A), mass)


# Probability axioms and elementary consequences.
assert P(omega) == 1
assert P(frozenset()) == 0
assert all(P(A) >= 0 for A in events)

for A in events:
    assert P(omega - A) == 1 - P(A)

    for B in events:
        assert P(A | B) == P(A) + P(B) - P(A & B)

        assert P(A & B) <= min(P(A), P(B))
        assert P(A & B) >= max(
            Fraction(0, 1),
            P(A) + P(B) - 1,
        )

        if A <= B:
            assert P(A) <= P(B)
            assert P(B - A) == P(B) - P(A)

        if A.isdisjoint(B):
            assert P(A | B) == P(A) + P(B)

        assert P(A ^ B) == (
            P(A) + P(B) - 2 * P(A & B)
        )


# Inclusion–exclusion example.
O12 = frozenset(range(1, 13))
A2 = frozenset(x for x in O12 if x % 2 == 0)
A3 = frozenset(x for x in O12 if x % 3 == 0)
A4 = frozenset(x for x in O12 if x % 4 == 0)

lhs = Fraction(len(A2 | A3 | A4), 12)
rhs = (
    Fraction(len(A2), 12)
    + Fraction(len(A3), 12)
    + Fraction(len(A4), 12)
    - Fraction(len(A2 & A3), 12)
    - Fraction(len(A2 & A4), 12)
    - Fraction(len(A3 & A4), 12)
    + Fraction(len(A2 & A3 & A4), 12)
)
assert lhs == rhs == Fraction(2, 3)


# Proportional-mass example.
weights = [1, 2, 3, 4]
probabilities = [
    Fraction(w, sum(weights))
    for w in weights
]
assert sum(probabilities, Fraction(0, 1)) == 1
assert probabilities[2] + probabilities[3] == Fraction(7, 10)


# Geometric normalization via partial sum plus exact remainder.
for q in [Fraction(1, 2), Fraction(2, 3), Fraction(3, 4)]:
    for N in range(1, 10):
        partial = sum(
            (1 - q) * q ** n
            for n in range(N)
        )
        remainder = q ** N
        assert partial + remainder == 1


# Historical problem.
de_mere_one = 1 - Fraction(5, 6) ** 4
de_mere_two = 1 - Fraction(35, 36) ** 24

assert de_mere_one == Fraction(671, 1296)
assert de_mere_one > Fraction(1, 2)
assert de_mere_two < Fraction(1, 2)

show_result(
    "All Chapter 3 automatic checks passed",
    r"P(A\cup B)=P(A)+P(B)-P(A\cap B)",
    r"\max\{0,P(A)+P(B)-1\}\le P(A\cap B)\le\min\{P(A),P(B)\}",
    r"P(A\triangle B)=P(A)+P(B)-2P(A\cap B)",
    r"P_{\mathrm{de\ Mere},1}>\frac12>P_{\mathrm{de\ Mere},2}",
    note=(
        "Exact finite identities, inclusion--exclusion, point-mass "
        "normalization, geometric normalization and the de Méré "
        "probabilities all passed."
    ),
)


## 22. Chapter map

| Chapter idea | Computational representation |
|---|---|
| probability measure | exact finite point masses |
| probability axioms | normalization, non-negativity and disjoint additivity |
| complement / monotonicity | exhaustive finite event checks |
| probability zero / one | explicit non-empty null event |
| addition rule | decomposition into three disjoint pieces |
| Fréchet bounds | sharp four-point construction |
| inclusion--exclusion | exact divisibility example |
| Boole's inequality | interactive dependence-free union bound |
| continuity | increasing and decreasing geometric-mass events |
| first Borel--Cantelli lemma | shrinking tail-sum bounds |
| finite/countable models | normalization and point-mass construction |
| equally likely finite model | ordered-pair dice counting |
| calibration and model risk | loaded-die and regime-change simulations |
| de Méré problem | exact complete-sequence counting |
| capped transformation | event translation and transformed masses |

The guiding principle is:

> Probability theory specifies mathematically coherent models. Whether a particular model is scientifically appropriate is a separate modelling question.
